# ML Capstone — Refresh / Content Opportunity Scoring

**Author:** Soham Shukla  
**Lane:** Refresh / Content Opportunity Scoring  
**Repo:** https://github.com/Soham334/flyrank-ml-internship

This capstone uses the existing Week 1–5 work as its foundation. Final metrics are generated from the anonymized dataset at run time; no results are invented or hard-coded.

## 1. Question

**Research question:** Can an interpretable ML ranking model identify content that is more likely to be declining, and does it improve ranking beyond the Week-4 transparent refresh-review baseline?

**Decision:** prioritize content for human refresh review.

**Unit:** one pseudonymized content item. **Output:** ranked review queue. **Wrong-call cost:** false positives consume review time; false negatives can leave declining content unreviewed.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

SEED = 42
DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run from repo root with the permitted anonymized dataset available locally.')
df = pd.read_csv(DATA_PATH)
print('Raw rows:', len(df))

## 2. Data safety

Use pre-decision performance/content fields as features. Exclude `trend_direction`, `trend_pct`, product-generated health/action flags, and pseudonymous IDs from predictive features. IDs may be used only for grouping. No client names, domains, URLs, private queries, or credentials belong in public outputs.

In [ ]:
numeric_features = [
    'search_volume','competition','cpc','word_count','char_count',
    'impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d',
    'engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
    'days_with_impressions','days_with_sessions','impressions_last_30d',
    'clicks_last_30d','sessions_last_30d','impressions_prev_30d',
    'clicks_prev_30d','sessions_prev_30d','content_age_days',
    'age_tier_order','days_since_last_update','ctr','avg_position',
    'engagement_rate','scroll_rate','ai_traffic_pct'
]
categorical_features = [
    'competition_level','content_type','main_intent','provider_used',
    'model_used','age_tier','freshness_tier','word_count_tier',
    'char_count_tier','impression_tier','position_tier'
]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
for c in numeric_features:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in categorical_features:
    df[c] = df[c].fillna('unknown').astype(str)
if 'trend_direction' not in df.columns:
    raise ValueError('trend_direction is required as the evaluation outcome.')
df['y'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
for src, dst in [('impressions_90d','log_impressions_90d'),('clicks_90d','log_clicks_90d'),('sessions_90d','log_sessions_90d'),('ai_sessions_90d','log_ai_sessions_90d')]:
    if src in df.columns:
        df[dst] = np.log1p(df[src].fillna(0).clip(lower=0))
extra_logs = [c for c in ['log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d'] if c in df.columns]
model_numeric = numeric_features + extra_logs
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
if 'content_id' in df.columns:
    df = df.drop_duplicates('content_id').reset_index(drop=True)
print('Usable rows:', len(df))
print('Base rate:', round(df['y'].mean(), 4))

## 3. Baseline

Week-4 baseline: flag an item for `REFRESH_REVIEW` when it is sufficiently stale (`days_since_last_update > 90`) and still has meaningful visibility (`impressions_90d >= 100`). It is transparent, cheap to reproduce, and provides the same decision context as the model.

## 4. Model / analysis

Use regularized logistic regression with standardized numeric features and one-hot categorical features. Target: `trend_direction == 'down'`. This is deliberately interpretable and is evaluated against the transparent baseline on the same held-out data.

In [ ]:
X = df[model_numeric + categorical_features].copy()
y = df['y'].copy()
if 'client_id' in df.columns:
    groups = df['client_id'].astype(str)
else:
    groups = pd.Series(np.arange(len(df)), index=df.index).astype(str)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
prep = ColumnTransformer([('num', num_pipe, model_numeric), ('cat', cat_pipe, categorical_features)])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED))])
model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:,1]
test = df.iloc[test_idx].copy()
test['model_score'] = model_score
test['baseline_score'] = ((test['days_since_last_update'].fillna(0) > 90) & (test['impressions_90d'].fillna(0) >= 100)).astype(int)
print('Train:', len(train_idx), 'Test:', len(test_idx), 'Test base rate:', round(y_test.mean(),4))

## 5. Evaluation — same split, same outcome

Report ROC-AUC and average precision, plus precision@K. Always show the test-set base rate next to precision@K so a high score is not mistaken for strong discrimination.

In [ ]:
def p_at_k(y_true, scores, k):
    frame = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(scores)})
    return float(frame.sort_values('score', ascending=False).head(min(k,len(frame)))['y'].mean())
metrics = {
  'seed': SEED,
  'split': 'GroupShuffleSplit by client_id when available; test_size=0.20',
  'train_rows': int(len(train_idx)), 'test_rows': int(len(test_idx)),
  'test_base_rate': float(y_test.mean()),
  'model_roc_auc': float(roc_auc_score(y_test, model_score)) if y_test.nunique()>1 else None,
  'baseline_roc_auc': float(roc_auc_score(y_test, test['baseline_score'])) if y_test.nunique()>1 else None,
  'model_average_precision': float(average_precision_score(y_test, model_score)) if y_test.nunique()>1 else None,
  'baseline_average_precision': float(average_precision_score(y_test, test['baseline_score'])) if y_test.nunique()>1 else None,
  'model_precision_at_20': p_at_k(y_test, model_score, 20),
  'baseline_precision_at_20': p_at_k(y_test, test['baseline_score'], 20)
}
Path('work/outputs').mkdir(parents=True, exist_ok=True)
Path('work/figures').mkdir(parents=True, exist_ok=True)
Path('work/outputs/capstone_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

## 6. Interpretation and error analysis

Interpret coefficients directionally. Do not turn associations into causal claims. Review false positives and false negatives as examples of where human judgment remains necessary.

In [ ]:
test['predicted'] = (test['model_score'] >= 0.5).astype(int)
test['error_type'] = np.select([
    (test['predicted']==1)&(test['y']==0),
    (test['predicted']==0)&(test['y']==1)
], ['false_positive','false_negative'], default='correct')
print(test['error_type'].value_counts().to_string())
try:
    names = model.named_steps['prep'].get_feature_names_out()
    coef = model.named_steps['clf'].coef_[0]
    effects = pd.DataFrame({'feature':names,'coefficient':coef})
    print(effects.reindex(effects.coefficient.abs().sort_values(ascending=False).index).head(15).to_string(index=False))
except Exception as e:
    print('Coefficient inspection unavailable:', e)

## 7. Ranked recommendations

1. Review the highest model-score items first, especially when they also meet the stale+visible baseline.
2. Use the baseline reason code as an interpretable check on model recommendations.
3. Send model/baseline disagreements to human review rather than auto-refreshing content.
4. Re-run the ranking when the data window changes or the outcome base rate shifts materially.

**Framing:** observed/measured/directional decision support only. No causal or Google-algorithm claims.

In [ ]:
queue = test.copy()
cut = queue['model_score'].quantile(0.80)
queue['reason_code'] = np.select([
    (queue['model_score']>=cut)&(queue['baseline_score']==1),
    queue['model_score']>=cut,
    queue['baseline_score']==1
], ['MODEL_HIGH_AND_STALE_VISIBLE','MODEL_HIGH','STALE_VISIBLE'], default='REVIEW_IF_NEEDED')
queue = queue.sort_values(['model_score','impressions_90d'], ascending=False)
cols=[c for c in ['model_score','baseline_score','reason_code','impressions_90d','days_since_last_update','content_age_days'] if c in queue]
queue[cols].head(20).to_json('work/outputs/capstone_top20.json', orient='records', indent=2)
display(queue[cols].head(20))
plt.figure(figsize=(7,5))
plt.bar(['Baseline AP','Model AP'], [metrics['baseline_average_precision'], metrics['model_average_precision']])
plt.ylabel('Average precision'); plt.title('Model vs baseline on the same held-out split')
plt.tight_layout(); plt.savefig('work/figures/model_vs_baseline_average_precision.png', dpi=160); plt.show()

## 8. Reproducibility

Random seed: **42**. Install with `pip install pandas numpy scikit-learn matplotlib`. Run the notebook from the repository root with the permitted anonymized dataset available locally. The dataset itself must not be committed. Metrics JSON and generated paper figures are the reproducibility receipts.

## 9. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

This analysis is observational and intended for decision support. It does not establish causality, identify clients/domains/queries, or claim to predict Google's algorithm.

## ML-12 closing material

**5-minute demo:** question → data safety → baseline → model → grouped evaluation → results → ranked queue → limitations.

**Social-post cut:** Built a reproducible content-refresh opportunity ranking workflow using an interpretable baseline and supervised model, with grouped validation and explicit leakage controls. The output is a human-review queue, not an automatic refresh decision.

**Employer-facing summary:** I built a reproducible ML workflow that turns anonymized content-performance signals into a ranked refresh-review queue. I compared an interpretable baseline with a supervised model under grouped validation and documented leakage controls, errors, and limitations. The result is a public-safe research artifact designed for decision support.